In [5]:
# importe la bibliotheque de gestion et analyse des tableaux de donnees
import pandas as pd

# importe le sous module pylot pour les visualisations
import matplotlib.pyplot as plt

In [6]:
# charge le fichier pdf dans un DataFrame
df = pd.read_csv('../data/exoplanets_raw.csv', comment='#', low_memory=False)

# affiche le nb de lignes et de colonne
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
# affiche les premieres lignes du DataFrame
df.head()

Rows: 39443
Columns: 100


,pl_name,hostname,default_flag,sy_snum,sy_pnum,discoverymethod,disc_year,disc_facility,soltype,pl_controv_flag,...,pl_pubdate,releasedate,Unnamed: 92,Unnamed: 93,Unnamed: 94,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99
0,11 Com b,11 Com,1,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2023-08,2023-09-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2008-01,2014-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11 Com b,11 Com,0,2,1,Radial Velocity,2007.0,Xinglong Station,Published Confirmed,0,...,2011-08,2014-07-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11 UMi b,11 UMi,0,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,2009-10,2014-05-14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11 UMi b,11 UMi,1,1,1,Radial Velocity,2009.0,Thueringer Landessternwarte Tautenburg,Published Confirmed,0,...,2017-03,2018-09-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# definit une liste de noms de colonnes a conserver
useful_columns = [
    'pl_name', 'disc_year', 'discoverymethod',
    'pl_rade', 'pl_bmasse', 'pl_orbper',
    'sy_dist', 'pl_eqt', 'st_teff', 'sy_snum'
]

# reduit la dataframe pour ne garder que ces colonnes
df = df[useful_columns]
# affiche la nouvelle forme
print(df.shape)
df.head()

(39443, 10)


,pl_name,disc_year,discoverymethod,pl_rade,pl_bmasse,pl_orbper,sy_dist,pl_eqt,st_teff,sy_snum
0,11 Com b,2007.0,Radial Velocity,NaN,4914.898486,323.21,93.1846,NaN,4874,2
1,11 Com b,2007.0,Radial Velocity,NaN,6165.600000,326.03,93.1846,NaN,4742,2
2,11 Com b,2007.0,Radial Velocity,NaN,5434.700000,NaN,93.1846,NaN,NaN,2
3,11 UMi b,2009.0,Radial Velocity,NaN,3337.070000,516.22,125.321,NaN,4340,1
4,11 UMi b,2009.0,Radial Velocity,NaN,4684.814200,516.21997,125.321,NaN,4213,1


In [8]:
# supprime les doublons dans pl_name et conserve la premiere occurrence
df = df.drop_duplicates(subset='pl_name', keep='first')

# qffiche le nombre de lignes restantes
print(f"Unique planets: {df.shape[0]}")

Unique planets: 6128


In [9]:
# calcule pour chaque colonne, le nombre de valeurs manquantes
df.isnull().sum()

pl_name               0
disc_year             1
discoverymethod       0
pl_rade            2400
pl_bmasse          3770
pl_orbper           563
sy_dist             128
pl_eqt             3415
st_teff            1047
sy_snum               0
dtype: int64

In [10]:
# produit un resume statistiques des colonnes
df.describe()

,disc_year,pl_rade,pl_bmasse,pl_eqt,sy_snum
count,6127.000000,3728.000000,2358.000000,2713.000000,6128.000000
mean,2016.954464,5.274184,825.025167,846.276786,1.103296
std,4.945150,70.275857,1659.862590,489.588319,0.342233
min,1992.000000,-0.009000,-0.017040,-35.000000,1.000000
25%,2014.000000,1.439599,8.902500,516.000000,1.000000
50%,2016.000000,2.300000,162.092488,779.150000,1.000000
75%,2021.000000,3.710000,794.553263,1134.310000,1.000000
max,2026.000000,4282.980000,22934.497849,3053.000000,4.000000


In [11]:
# convertit df en csv
df.to_csv('../data/exoplanets_clean.csv', index=False)
print("File saved!")

File saved!


In [ ]:
from sqlalchemy import create_engine

# crée une connexion vers une base de donees SQLite. Le fichier exoplanets.db est stocké dans le dossier data.
engine = create_engine('sqlite:///../data/exoplanets.db')

# prends le dataframe pandas (les planètes nettoyées) et l'insère dans un table SQL
df.to_sql('exoplanets', engine, if_exists='replace', index=False)

# En résumé, cette celulle convertit le CSV nettoyé en base de données SQL pour pouvoir faire des
# requêtes dessus. C'est le pont entre pandas et SQL
print("Database created!")

Database created!


In [ ]:
# Combien de planètes ont été découvertes par chaque méthode ?

query = """
SELECT discoverymethod, COUNT(*) as total
FROM exoplanets
GROUP BY discoverymethod
ORDER BY total DESC
"""

# éxecute la requête et convertit le résultat en DataFrame pandas
result = pd.read_sql(query, engine)
result

# Ce tableau montre le nombre de planètes découvertes par méthode de détection :

# Transit avec 4511 planètes (73%) - La méthode dominante. Elle consiste à détecter la baisse de
# luminosité d'une étoile quand une planète passe devant elle. C'est la méthode du télescope
# Kepler - ce qui explique le nb énorme

# Radial Velocity qvec 1171 planètes (19%) - La méthode historique. On mesure les petits mouvements
# d'une étoile causés par la gravité d'une planète. C'est la méthode principale avant Kepler.

# Microlensing avec 270 planètes (4%) - On utilise la déformation de la lumière d'une étoile
# lointaine par la gravité d'une planète. Rare et difficile à observer.

# Imaging avec 94 planètes (1.5%) - On photographie directement la planète. Très difficile car 
# les planètes sont beaucoup moins lumineuses que leur étoile.

# Les autres méthodes - Transit Timing Variations, Eclipse Timing Variations, Pulsar Timing...
# représentant moins de 1% chacune. Ce sont des méthodes très spécialisées.

# L'insight => 92% des exoplanètes connues ont été découvertes par seulement é méthodes :
# Transit et Radial Velocity. Sans Kepler, on en connaîtrait 10 fois moins! 

,discoverymethod,total
0,Transit,4511
1,Radial Velocity,1171
2,Microlensing,270
3,Imaging,94
4,Transit Timing Variations,39
5,Eclipse Timing Variations,17
6,Orbital Brightness Modulation,9
7,Pulsar Timing,8
8,Astrometry,6
9,Pulsation Timing Variations,2


In [ ]:
# Combien de planètes ont été découvertes chaque année ?

query = """
SELECT disc_year, COUNT(*) as total
FROM exoplanets
WHERE disc_year IS NOT NULL
GROUP BY disc_year
ORDER BY disc_year ASC
"""

result2 = pd.read_sql(query, engine)
result2

# Ce tableau montre l'évolution des découvertes d'exoplanètes année par année

# 1992-2008 : L'ère pionnière - Quelques dizaines de planètes par an maximum. Les astronomes
# utilisaient principalement la Radial Velocity, une méthode lente et difficile. En 14 ans, on
# découvrait à peine plus de 60 planètes par an

# 2009 : Le tournant - 91 découvertes. C'est l'année du lancement du télescope Kepler. 
# Les effets ne se font pas encore sentir car il faut du temps pour analyser les données.

# 2014 : La première explosion - 869 planètes en une seule année ! Kepler publie ses premières
# grandes séries de résultats. C'est une multiplication par 10 en quelques années.

# 2016 : Le pic historique - 1496 planètes, le record absolu. Kepler est en pleine forme et 
# publie ses données en masse.

# 2017-2020: La stabilisation - Entre 150 et 315 par an. Kepler commence à s'essouffler mais
#  le télescope TESS prend le relais.

# 2021: Nouveau pic - 564 planètes. TESS publie ses premières grandes séries de résultats

# 2026 - 44 planètes - On est seulement en mars 2026, donc ce chiffre devrait encore augmenter !

# L'insight => Sans les télescopes spatiaux Kepler et TESS, on connaîtrait 10 fois moins
# d'exoplanètes. La technologie est un vrai moteur de la découverte 

# Jusqu'en 2010, on découvrait moins de 100 planètes par an. Depuis Kepler, le nombre a explosé, atteignant un pic en 2014, 2016 et 2021

,disc_year,total
0,1992.0,2
1,1994.0,1
2,1995.0,1
3,1996.0,6
4,1997.0,1
5,1998.0,6
6,1999.0,13
7,2000.0,16
8,2001.0,12
9,2002.0,29


In [ ]:
# Corriger le type de sy_dist (distance en parsecs)
# la colonne sy_dist avait été mal interprétée par pandas comme du TEXT au lieu de nb FLOAT
# Ici on convertit la colonne en nombres, coerce -> remplace par NaN si une valeur ne peut etre
# convertie
df['sy_dist'] = pd.to_numeric(df['sy_dist'], errors='coerce')

# Remettre à jour la base SQLite
df.to_sql('exoplanets', engine, if_exists='replace', index=False)
print("Database updated!")

Database updated!


In [ ]:
# Quelles sont les 10 exoplanètes les plus proches de la Terre ?

query = """
SELECT pl_name, discoverymethod, sy_dist, disc_year
FROM exoplanets
WHERE sy_dist > 0
ORDER BY sy_dist ASC
LIMIT 10
"""

result3 = pd.read_sql(query, engine)
result3

# ce tableau montre les 10 exoplanètes les plus proches de la Terre.

# WASP-151 b — 0.30 parsecs — Suspicieusement proche ! On a vu plus tôt que cette planète a 
# 3 entrées dans le dataset avec des distances très différentes (0.30, NaN, et 453 parsecs). 
# C'est clairement une erreur de saisie dans la base NASA — la vraie distance est probablement 
# 453 parsecs.

# Proxima Cen b et d — 1.30 parsecs — Les vraies exoplanètes les plus proches de la Terre ! 
# Elles orbitent autour de Proxima Centauri, l'étoile la plus proche du Soleil. 1.30 parsecs = 
# environ 4.2 années-lumière. Proxima Cen b est dans la zone habitable de son étoile !

# Barnard b, c, d, e — 1.82 parsecs — 4 planètes autour de l'étoile de Barnard, la deuxième 
# étoile la plus proche du Soleil. Toutes découvertes très récemment en 2024-2025 !

# WASP-82 b, WASP-24 b, HAT-P-64 b — Des Hot Jupiters (géantes gazeuses très proches de 
# leur étoile) découverts par Transit.

# L'insight => Les étoiles les plus proches de nous ont toutes des planètes confirmées. La
# galaxie est probablement remplie de planètes

# Convention de nommage de NASA (pl_name) :
# Le nom de l'étoile hôte + une lettre pour la planète
# ex: Proxima Cen = nom de l'étoile
# Proxima Cen b = première planète découverte autour de cette étoile
# hostname est seulement le nom de l'étoile

,pl_name,discoverymethod,sy_dist,disc_year
0,WASP-151 b,Transit,0.306690,2017.0
1,Proxima Cen b,Radial Velocity,1.301190,2016.0
2,Proxima Cen d,Radial Velocity,1.301190,2025.0
3,Barnard b,Radial Velocity,1.826550,2024.0
4,Barnard c,Radial Velocity,1.826550,2025.0
5,Barnard d,Radial Velocity,1.826550,2025.0
6,Barnard e,Radial Velocity,1.826550,2025.0
7,WASP-82 b,Transit,1.893836,2015.0
8,WASP-24 b,Transit,2.343286,2010.0
9,HAT-P-64 b,Transit,2.431273,2021.0


In [ ]:
# Quels est la masse et le rayon moyen des planètes pour chaque méthode de détection ?

query = """
SELECT discoverymethod,
COUNT(*) AS total,
ROUND(AVG(pl_bmasse), 2) AS avg_mass,
ROUND(AVG(pl_rade), 2) AS avg_radius
FROM exoplanets
WHERE pl_bmasse IS NOT NULL
AND pl_rade IS NOT NULL
GROUP BY discoverymethod
ORDER BY total DESC
"""

# Ce tableau montre les moyennes de masse et rayon par méthode de détection.

# Transit — 820 planètes
# Masse moyenne : 272 masses terrestres
# Rayon moyen : 6.30 rayons terrestres
# Détecte des planètes de taille variée, des petites Super-Terres aux grosses géantes gazeuses

# Imaging — 20 planètes
# Masse moyenne : 4287 masses terrestres
# Rayon moyen : 24.38 rayons terrestres
# Les planètes les plus massives de loin ! Normal — on ne peut photographier directement que 
# des géantes gazeuses énormes, les petites planètes sont invisibles à côté de leur étoile

# Radial Velocity — 162 planètes
# Masse moyenne : 16 masses terrestres
# Rayon moyen : 3.31 rayons terrestres
# Détecte des planètes plus petites que Imaging mais plus grandes que la Terre en moyenne

# L'insight => Chaque méthode a un biais de détection. Imaging ne voit que les géantes, 
# Transit voit tout, Radial Velocity préfère les planètes massives. Ce n'est pas que l'univers 
# est rempli de géantes — c'est qu'on a du mal à détecter les petites ! 🚀

result_avg = pd.read_sql(query, engine)
result_avg


,discoverymethod,total,avg_mass,avg_radius
0,Transit,820,272.96,6.30
1,Imaging,20,4287.34,24.38
2,Radial Velocity,16,216.10,3.31
3,Orbital Brightness Modulation,2,0.55,0.81
4,Transit Timing Variations,1,8.50,1.56


In [ ]:
# Cette cellule crée une nouvelle table dans la base SQLite spécialement pour pousser les analyses
# vers les systèmes stellaires.

# recharge le fichier CSV brut depuis le début, avant tout nettoyage
df_full = pd.read_csv('../data/exoplanets_raw.csv', comment='#', low_memory=False)

# garde les même colonnes utiles qu'avant mais en ajoutant hostname - le nom de l'étoile hôte,
# indispensable pour grouper les planètes par système
df_full = df_full[['pl_name', 'hostname', 'disc_year', 'discoverymethod',
                    'pl_rade', 'pl_bmasse', 'pl_orbper',
                    'sy_dist', 'pl_eqt', 'st_teff', 'sy_snum']]

# ici la diff avec l'autre table, ici on dédoublonne sur hostname + pl_name ensemble -> 1 entrée
# par planète par étoile -> on s'assure qu'une planète n'apparaît qu'une fois par système stellaire
df_full = df_full.drop_duplicates(subset=['hostname', 'pl_name'], keep='first')

# créé une nouvelle table séparée appelée exoplanets_systems sans toucher à l'autre table
# plusieurs tables pour différents besoins !
df_full.to_sql('exoplanets_systems', engine, if_exists='replace', index=False)
print(f"Table created with {len(df_full)} rows !")

Table created with 6128 rows !


In [ ]:
# Quelles sont les 10 étoiles qui ont plus de 3 planètes confirmées ?

query = """
SELECT 
    hostname,
    COUNT(*) as nb_planets
FROM exoplanets_systems
GROUP BY hostname
HAVING nb_planets > 3
ORDER BY nb_planets DESC
LIMIT 10
"""

result_having = pd.read_sql(query, engine)
result_having

# Ce tableau montre les 10 étoiles avec le plus de planètes confirmées

# KOI-351 — 8 planètes
# Le système avec le plus de planètes confirmées de notre dataset ! Aussi appelé Kepler-90, 
# c'est le seul système connu avec autant de planètes que notre Système Solaire (8 également). 
# Découvert par Kepler.

# TRAPPIST-1 — 7 planètes
# Le système le plus célèbre pour la recherche de vie extraterrestre ! 3 de ses planètes (e, f, g) 
# sont dans la zone habitable de leur étoile. C'est une étoile naine ultra-froide à seulement 40 
# années-lumière de la Terre.

# TOI-178 et TOI-1136 — 6 planètes
# Découverts par le télescope TESS (successeur de Kepler). TOI-178 est particulièrement intéressant 
# car ses planètes sont en résonance orbitale — leurs orbites sont synchronisées comme une 
# chorégraphie !

# Kepler-80, Kepler-20, Kepler-11 — 6 planètes
# Trois systèmes découverts par Kepler. Kepler-11 a été une découverte historique en 2011 — c'était 
# le premier système avec autant de planètes confirmées en transit simultané.

# K2-138, HIP 41378, HD 34445 — 6 planètes
# Des systèmes découverts plus récemment par K2 (la mission secondaire de Kepler) et par Radial 
# Velocity.

# L'insight => Les systèmes multi-planètes ne sont pas rares ! Notre Système Solaire avec 8
# planètes n'est pas une exception dans la galaxie, KOI-351 le prouve !

,hostname,nb_planets
0,KOI-351,8
1,TRAPPIST-1,7
2,TOI-178,6
3,TOI-1136,6
4,Kepler-80,6
5,Kepler-20,6
6,Kepler-11,6
7,K2-138,6
8,HIP 41378,6
9,HD 34445,6


In [ ]:
# Quelles sont les planètes les plus grandes que la moyenne ?

query = """
SELECT pl_name, discoverymethod, pl_rade
FROM exoplanets
WHERE pl_rade > (SELECT AVG(pl_rade) FROM exoplanets)
ORDER BY pl_rade DESC
LIMIT 10
"""


result_sub = pd.read_sql(query, engine)
result_sub

# Ce tableau montre les 10 planètes avec le plus grand rayon, toutes au dessus de la moyenne

# Kepler-1999 b — 4282 rayons terrestres
# C'est un outlier énorme — probablement une erreur dans le dataset ou une étoile mal classifiée 
# comme planète. Une vraie planète ne peut pas faire 4282 fois la Terre !

# Kepler-1983 b et V2376 Ori b — ~88 rayons terrestres
# Des géantes gazeuses massives, similaires à Jupiter qui fait environ 11 rayons terrestres
# — mais en beaucoup plus grosses.

# HD 100546 b et GQ Lup b — découvertes par Imaging
# Ce sont des planètes encore en formation ! On peut les photographier directement car elles
# sont encore très chaudes et lumineuses. C'est pour ça qu'elles sont si grandes.

# PDS 70 b — 30 rayons terrestres
# Une des premières planètes en formation jamais photographiées directement. Une découverte
# historique en 2018 !

# L'insight => Les planètes découvertes par Imaging dominent ce top 10 malgré leur nombre très
# faible. Cela confirme ce qu'on avait vu dans la requête AVG - Imaging ne détecte que les
# géantes

,pl_name,discoverymethod,pl_rade
0,Kepler-1999 b,Transit,4282.98000
1,Kepler-1983 b,Transit,88.52000
2,V2376 Ori b,Imaging,87.20587
3,HD 100546 b,Imaging,77.34210
4,Kepler-1935 b,Transit,64.89000
5,GQ Lup b,Imaging,51.56140
6,Kepler-419 b,Transit,30.64000
7,PDS 70 b,Imaging,30.48848
8,Kepler-359 d,Transit,30.34000
9,KOI-2513.01,Transit,28.47000
